# Fink/LSST — Angular Correlation Functions in Time Slices for a Single DDF

This notebook computes the **two-point angular correlation function** $w(\theta)$
in configurable **time slices** for a single LSST Deep Drilling Field.

Data are loaded from parquet files produced by notebook `01c_fink_dipoles_per_ddf.ipynb`
(directory `data_DIPOLES_01c/`).  **No Fink API calls are made here.**

## Scientific goals

For a single DDF and for each time slice:

1. **Auto-correlation of all alerts** — is the spatial distribution uniform within a season?
2. **Auto-correlation of dipole alerts** — do dipoles cluster, and does clustering evolve with time
   (e.g. as template quality improves during the survey)?
3. **Cross-correlation non-dipoles × dipoles** — do dipoles trace the same sky as non-dipoles?

Each of the three correlation types is shown as a **single summary figure** in which
each time slice is one coloured curve.  Slices with fewer than `N_MIN_POINTS` alerts
are skipped silently.

## Method

Landy–Szalay estimator with a uniform circular random catalogue:

$$w(\theta) = \frac{DD(\theta) - 2\,DR(\theta) + RR(\theta)}{RR(\theta)}$$

TreeCorr is used when available; a NumPy/KDTree flat-sky fallback is provided.

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-23
- last update : 2026-05-23

## 1. Imports & configuration

In [ ]:
import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.time import Time
from scipy.spatial import KDTree

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

try:
    import treecorr

    HAS_TREECORR = True
    print(f"treecorr version : {treecorr.__version__}  ✓")
except ImportError:
    HAS_TREECORR = False
    print("treecorr NOT found — using NumPy KD-tree fallback.")
    print("Install with:  pip install treecorr")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# USER PARAMETERS  ← edit here
# ═══════════════════════════════════════════════════════════════════════════════

# ── Which field to analyse ────────────────────────────────────────────────────
FIELD_NAME = "COSMOS"  # must match a key in DEEP_FIELDS below

# ── Time-slice configuration ──────────────────────────────────────────────────
SLICE_DAYS = 60  # width of each time slice in days (e.g. 60 = ~2 months)

# ── Minimum number of points to attempt a correlation computation ─────────────
N_MIN_POINTS = 50  # slices with fewer data points are skipped

# ── Angular binning ───────────────────────────────────────────────────────────
THETA_MIN_DEG = 0.001  # minimum angular separation (degrees)
THETA_MAX_DEG = 0.9  # maximum angular separation (degrees)
N_THETA_BINS = 20  # number of log-spaced bins

# ── Random catalogue ─────────────────────────────────────────────────────────
N_RANDOM = 5000  # number of random points per correlation call
RANDOM_SEED = 42

# ═══════════════════════════════════════════════════════════════════════════════

# ── Input parquet directory (from notebook 01c) ───────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_02b"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Field          : {FIELD_NAME}")
print(f"Slice width    : {SLICE_DAYS} days")
print(f"Min points     : {N_MIN_POINTS}")
print(f"Input data     : {os.path.abspath(DIR_DATA_IN)}")
print(f"Output data    : {os.path.abspath(DIR_DATA)}")
print(f"Figures        : {os.path.abspath(DIR_FIGS)}")

# ── DDF catalogue ─────────────────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}
CONE_RADIUS_DEG = 1.0

assert FIELD_NAME in DEEP_FIELDS, f"{FIELD_NAME} not in DEEP_FIELDS"
RA_CENTER, DEC_CENTER = DEEP_FIELDS[FIELD_NAME]
print(f"Field centre   : RA={RA_CENTER:.4f}  Dec={DEC_CENTER:.4f}")

# ── Plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str):
    """Save figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Load the parquet file for the selected DDF

In [ ]:
pq_path = os.path.join(DIR_DATA_IN, f"{FIELD_NAME}_alerts.parquet")
assert os.path.exists(pq_path), f"Parquet not found: {pq_path}\nRun notebook 01c first."

df_full = pd.read_parquet(pq_path)

# ── Numeric coercions ─────────────────────────────────────────────────────────
for col in ("r:ra", "r:dec", "r:midpointMjdTai"):
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors="coerce")

# ── Boolean dipole flag ───────────────────────────────────────────────────────
if "r:isDipole" in df_full.columns:
    df_full["r:isDipole"] = (
        df_full["r:isDipole"]
        .map(
            lambda v: (
                True
                if str(v).strip().lower() in ("true", "1", "yes")
                else False
                if str(v).strip().lower() in ("false", "0", "no")
                else pd.NA
            )
        )
        .astype("boolean")
    )

df_full = df_full.dropna(subset=["r:ra", "r:dec", "r:midpointMjdTai"])

n_tot = len(df_full)
n_dip = int(df_full["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df_full.columns else 0
mjd_lo = df_full["r:midpointMjdTai"].min()
mjd_hi = df_full["r:midpointMjdTai"].max()

# Convert MJD limits to calendar dates for display
date_lo = Time(mjd_lo, format="mjd", scale="tai").strftime("%Y-%m-%d")
date_hi = Time(mjd_hi, format="mjd", scale="tai").strftime("%Y-%m-%d")

print(f"Field          : {FIELD_NAME}")
print(f"Total alerts   : {n_tot:,}   ({n_dip:,} dipoles,  {n_tot - n_dip:,} non-dipoles)")
print(f"MJD range      : {mjd_lo:.2f} – {mjd_hi:.2f}")
print(f"Date range     : {date_lo}  →  {date_hi}")
print(f"Total timespan : {mjd_hi - mjd_lo:.1f} days")

## 3. Build time slices

The full MJD range of the parquet file is divided into contiguous windows
of `SLICE_DAYS` days.  For each slice we record:
- the MJD interval `[mjd_start, mjd_end)`
- the corresponding calendar dates (for plot labels)
- the sub-DataFrame restricted to that window

Slices with fewer than `N_MIN_POINTS` alerts **in total** are flagged as empty
and will be skipped in the correlation computation.

In [ ]:
def mjd_to_datestr(mjd: float) -> str:
    """Convert a single MJD (TAI) value to a 'YYYY-MM-DD' string."""
    return Time(float(mjd), format="mjd", scale="tai").strftime("%Y-%m-%d")


def build_time_slices(
    df: pd.DataFrame,
    slice_days: float,
    n_min: int,
    mjd_col: str = "r:midpointMjdTai",
    dipole_col: str = "r:isDipole",
) -> list[dict]:
    """
    Divide the MJD range of *df* into contiguous windows of *slice_days* days.

    Returns
    -------
    List of dicts, one per slice, with keys:
        idx          : integer slice index
        mjd_start    : float
        mjd_end      : float
        date_start   : str  'YYYY-MM-DD'
        date_end     : str  'YYYY-MM-DD'
        label        : str  short label for plot legend
        df_all       : DataFrame  — all alerts in this slice
        df_dip       : DataFrame  — dipole alerts in this slice
        df_ndip      : DataFrame  — non-dipole alerts in this slice
        skip         : bool  True if too few points
        skip_reason  : str   explanation when skip=True
    """
    mjd_min = df[mjd_col].min()
    mjd_max = df[mjd_col].max()
    edges = np.arange(mjd_min, mjd_max + slice_days, slice_days)

    slices = []
    for i in range(len(edges) - 1):
        t0, t1 = edges[i], edges[i + 1]
        mask = (df[mjd_col] >= t0) & (df[mjd_col] < t1)
        sub = df[mask].copy()

        mask_dip = (
            sub[dipole_col].fillna(False).astype(bool)
            if dipole_col in sub.columns
            else pd.Series(False, index=sub.index)
        )
        df_dip = sub[mask_dip]
        df_ndip = sub[~mask_dip]

        n_all = len(sub)
        skip = n_all < n_min
        reason = f"n_all={n_all} < N_MIN_POINTS={n_min}" if skip else ""

        slices.append(
            {
                "idx": i,
                "mjd_start": t0,
                "mjd_end": t1,
                "date_start": mjd_to_datestr(t0),
                "date_end": mjd_to_datestr(t1),
                "label": f"{mjd_to_datestr(t0)} → {mjd_to_datestr(t1)}",
                "df_all": sub,
                "df_dip": df_dip,
                "df_ndip": df_ndip,
                "skip": skip,
                "skip_reason": reason,
            }
        )

    return slices


time_slices = build_time_slices(df_full, SLICE_DAYS, N_MIN_POINTS)

print(f"\n{'Slice':>5}  {'Date range':>30}  {'N_all':>7}  {'N_dip':>6}  {'N_ndip':>7}  {'Status'}")
print("-" * 70)
for sl in time_slices:
    status = "SKIP  " + sl["skip_reason"] if sl["skip"] else "OK"
    print(
        f"{sl['idx']:>5}  {sl['label']:>30}  "
        f"{len(sl['df_all']):>7,}  "
        f"{len(sl['df_dip']):>6,}  "
        f"{len(sl['df_ndip']):>7,}  "
        f"{status}"
    )

n_active = sum(1 for sl in time_slices if not sl["skip"])
print(f"\nActive slices : {n_active} / {len(time_slices)}")

## 4. Correlation function engine

Same implementation as `02_fink_dipoles_uniformity.ipynb` — TreeCorr preferred,
NumPy/KDTree fallback.  Both return `{theta (deg), w, w_err}`.

In [ ]:
# ── Angular bins (log-spaced) ─────────────────────────────────────────────────
THETA_EDGES = np.logspace(
    np.log10(THETA_MIN_DEG),
    np.log10(THETA_MAX_DEG),
    N_THETA_BINS + 1,
)
THETA_CENTERS = np.sqrt(THETA_EDGES[:-1] * THETA_EDGES[1:])

print(f"Theta bins : {N_THETA_BINS}   [{THETA_MIN_DEG:.4f}, {THETA_MAX_DEG:.3f}] deg")


# ─────────────────────────────────────────────────────────────────────────────
# Random catalogue
# ─────────────────────────────────────────────────────────────────────────────
def make_random_catalogue(
    ra_center: float,
    dec_center: float,
    radius_deg: float,
    n: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Uniform random points inside a circular cone (flat-sky approx)."""
    cos_dec = np.cos(np.radians(dec_center))
    ra_rand = np.empty(n)
    dec_rand = np.empty(n)
    filled = 0
    while filled < n:
        needed = (n - filled) * 4 // 3 + 100
        dx = rng.uniform(-radius_deg, radius_deg, needed) / cos_dec
        dy = rng.uniform(-radius_deg, radius_deg, needed)
        dist = np.sqrt((dx * cos_dec) ** 2 + dy**2)
        mask = dist <= radius_deg
        take = min(mask.sum(), n - filled)
        ra_rand[filled : filled + take] = ra_center + dx[mask][:take]
        dec_rand[filled : filled + take] = dec_center + dy[mask][:take]
        filled += take
    return ra_rand, dec_rand


# ─────────────────────────────────────────────────────────────────────────────
# TreeCorr backend
# ─────────────────────────────────────────────────────────────────────────────
def _acf_treecorr(
    ra1,
    dec1,
    ra2,
    dec2,
    ra_rand,
    dec_rand,
    theta_edges,
    is_cross=False,
):
    """Landy-Szalay estimator via TreeCorr (spherical geometry)."""
    min_sep = float(theta_edges[0]) * 60.0
    max_sep = float(theta_edges[-1]) * 60.0
    nbins = len(theta_edges) - 1
    kw = dict(min_sep=min_sep, max_sep=max_sep, nbins=nbins, sep_units="arcmin", bin_slop=0.1)

    cat1 = treecorr.Catalog(ra=ra1, dec=dec1, ra_units="degrees", dec_units="degrees")
    catr = treecorr.Catalog(ra=ra_rand, dec=dec_rand, ra_units="degrees", dec_units="degrees")

    if not is_cross:
        dd = treecorr.NNCorrelation(**kw)
        dr = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        dd.process(cat1)
        dr.process(cat1, catr)
        rr.process(catr)
        xi_res = dd.calculateXi(rr=rr, dr=dr)
        theta_arcmin = np.exp(dd.meanlogr)
    else:
        cat2 = treecorr.Catalog(ra=ra2, dec=dec2, ra_units="degrees", dec_units="degrees")
        d1d2 = treecorr.NNCorrelation(**kw)
        d1r = treecorr.NNCorrelation(**kw)
        d2r = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        d1d2.process(cat1, cat2)
        d1r.process(cat1, catr)
        d2r.process(cat2, catr)
        rr.process(catr)
        xi_res = d1d2.calculateXi(rr=rr, dr=d1r, rd=d2r)
        theta_arcmin = np.exp(d1d2.meanlogr)

    # calculateXi returns (xi, varxi) in recent versions
    w = xi_res[0]
    w_err = np.sqrt(np.abs(xi_res[1]))
    return {"theta": theta_arcmin / 60.0, "w": w, "w_err": w_err}


# ─────────────────────────────────────────────────────────────────────────────
# NumPy/KDTree fallback (flat-sky)
# ─────────────────────────────────────────────────────────────────────────────
def _count_pairs_kdtree(ra1, dec1, ra2, dec2, theta_edges, cos_dec):
    """Count pairs in each angular bin using a KD-tree (flat sky)."""
    xy1 = np.column_stack([ra1 * cos_dec, dec1])
    xy2 = np.column_stack([ra2 * cos_dec, dec2])
    tree2 = KDTree(xy2)
    cum = np.zeros(len(theta_edges), dtype=np.float64)
    for k, r in enumerate(theta_edges):
        hits = tree2.query_ball_point(xy1, r, workers=-1)
        cum[k] = sum(len(h) for h in hits)
    return np.diff(cum)


def _acf_numpy(ra1, dec1, ra2, dec2, ra_rand, dec_rand, theta_edges, dec_center, is_cross=False):
    """Flat-sky Landy-Szalay estimator using KD-tree pair counts."""
    cos_dec = np.cos(np.radians(dec_center))
    n1, nr = len(ra1), len(ra_rand)

    if not is_cross:
        DD = _count_pairs_kdtree(ra1, dec1, ra1, dec1, theta_edges, cos_dec)
        DR = _count_pairs_kdtree(ra1, dec1, ra_rand, dec_rand, theta_edges, cos_dec)
        RR = _count_pairs_kdtree(ra_rand, dec_rand, ra_rand, dec_rand, theta_edges, cos_dec)
        DD_n = DD / max(n1 * (n1 - 1), 1)
        DR_n = DR / max(n1 * nr, 1)
        RR_n = RR / max(nr * (nr - 1), 1)
        raw_dd = DD
    else:
        n2 = len(ra2)
        D1D2 = _count_pairs_kdtree(ra1, dec1, ra2, dec2, theta_edges, cos_dec)
        D1R = _count_pairs_kdtree(ra1, dec1, ra_rand, dec_rand, theta_edges, cos_dec)
        D2R = _count_pairs_kdtree(ra2, dec2, ra_rand, dec_rand, theta_edges, cos_dec)
        RR = _count_pairs_kdtree(ra_rand, dec_rand, ra_rand, dec_rand, theta_edges, cos_dec)
        DD_n = D1D2 / max(n1 * n2, 1)
        DR_n = 0.5 * (D1R / max(n1 * nr, 1) + D2R / max(n2 * nr, 1))
        RR_n = RR / max(nr * (nr - 1), 1)
        raw_dd = D1D2

    with np.errstate(invalid="ignore", divide="ignore"):
        w = np.where(RR_n > 0, (DD_n - 2.0 * DR_n + RR_n) / RR_n, np.nan)
        w_err = np.where(raw_dd > 0, (1.0 + w) / np.sqrt(raw_dd + 1e-30), np.nan)

    return {"theta": THETA_CENTERS, "w": w, "w_err": w_err}


# ─────────────────────────────────────────────────────────────────────────────
# Public dispatcher
# ─────────────────────────────────────────────────────────────────────────────
def angular_correlation(
    ra1,
    dec1,
    ra2=None,
    dec2=None,
    n_min=N_MIN_POINTS,
    n_random=N_RANDOM,
    seed=RANDOM_SEED,
):
    """
    Compute Landy-Szalay w(theta) for the selected DDF.
    Uses RA_CENTER / DEC_CENTER from the notebook-level configuration.
    Returns None if there are not enough points.
    """
    is_cross = (ra2 is not None) and (dec2 is not None)
    if len(ra1) < n_min:
        return None
    if is_cross and len(ra2) < n_min:
        return None

    rng = np.random.default_rng(seed)
    ra_rand, dec_rand = make_random_catalogue(RA_CENTER, DEC_CENTER, CONE_RADIUS_DEG, n_random, rng)

    if HAS_TREECORR:
        return _acf_treecorr(ra1, dec1, ra2, dec2, ra_rand, dec_rand, THETA_EDGES, is_cross=is_cross)
    else:
        return _acf_numpy(
            ra1, dec1, ra2, dec2, ra_rand, dec_rand, THETA_EDGES, dec_center=DEC_CENTER, is_cross=is_cross
        )


print("Correlation engine ready.")
print(f"Backend : {'TreeCorr' if HAS_TREECORR else 'NumPy/KDTree (flat-sky fallback)'}")

## 5. Compute correlations for each time slice

For each active slice (not skipped) we compute:
- `acf_all`  : auto-correlation of all alerts
- `acf_dip`  : auto-correlation of dipole alerts only
- `xcf`      : cross-correlation non-dipoles × dipoles

Each entry is `None` if the sub-catalogue is too small.

In [ ]:
results_acf_all = {}  # slice_idx → result dict or None
results_acf_dip = {}
results_xcf = {}

for sl in time_slices:
    idx = sl["idx"]
    label = sl["label"]

    if sl["skip"]:
        print(f"  slice {idx:02d}  [{label}]  SKIPPED — {sl['skip_reason']}")
        results_acf_all[idx] = None
        results_acf_dip[idx] = None
        results_xcf[idx] = None
        continue

    ra_all = sl["df_all"]["r:ra"].values
    dec_all = sl["df_all"]["r:dec"].values
    ra_dip = sl["df_dip"]["r:ra"].values
    dec_dip = sl["df_dip"]["r:dec"].values
    ra_nd = sl["df_ndip"]["r:ra"].values
    dec_nd = sl["df_ndip"]["r:dec"].values

    print(
        f"  slice {idx:02d}  [{label}]  N_all={len(ra_all):,}  N_dip={len(ra_dip):,}  N_ndip={len(ra_nd):,}"
    )

    # ── (A) Auto-correlation all ──────────────────────────────────────────────
    results_acf_all[idx] = angular_correlation(ra_all, dec_all)

    # ── (B) Auto-correlation dipoles ─────────────────────────────────────────
    results_acf_dip[idx] = angular_correlation(ra_dip, dec_dip)

    # ── (C) Cross-correlation non-dipoles × dipoles ───────────────────────────
    results_xcf[idx] = angular_correlation(
        ra_nd,
        dec_nd,
        ra2=ra_dip,
        dec2=dec_dip,
    )

print("\nAll slices processed.")

## 6. Colour palette for time slices

We assign one maximally-distinct colour per active time slice using a
perceptually-uniform colourmap (`tab20` for up to 20 slices,
`hsv` for more).  Skipped slices have no colour.

In [ ]:
active_slices = [sl for sl in time_slices if not sl["skip"]]
n_active = len(active_slices)

# Choose a colourmap with maximally distinct hues
if n_active <= 10:
    cmap = cm.get_cmap("tab10", n_active)
elif n_active <= 20:
    cmap = cm.get_cmap("tab20", n_active)
else:
    cmap = cm.get_cmap("hsv", n_active)

# Map each active slice index to a colour
SLICE_COLOR = {}
for k, sl in enumerate(active_slices):
    SLICE_COLOR[sl["idx"]] = cmap(k / max(n_active - 1, 1))

# Markers cycle for extra disambiguation in B&W
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*", "h", "<", ">", "p"]
SLICE_MARKER = {sl["idx"]: MARKERS[k % len(MARKERS)] for k, sl in enumerate(active_slices)}

print(f"Active slices : {n_active}")
print(f"Colourmap     : {'tab10' if n_active <= 10 else 'tab20' if n_active <= 20 else 'hsv'}")
for sl in active_slices:
    print(f"  slice {sl['idx']:02d}  {sl['label']}  color={SLICE_COLOR[sl['idx']]}")

## 7. Summary figures

Three figures, one per correlation type.  Each figure shows **one curve per active
time slice**, colour-coded and with a marker for B&W readability.

A **shaded band** around $w=0$ helps assess which slices are consistent with uniformity.

In [ ]:
def plot_slices_summary(
    results_by_idx: dict,
    time_slices: list,
    title: str,
    figname: str,
    extra_info_fn=None,  # callable(sl) → str appended to legend label
) -> None:
    """
    Single-panel summary plot: one w(theta) curve per active time slice.

    Parameters
    ----------
    results_by_idx : dict  slice_idx → result dict (theta, w, w_err) or None
    time_slices    : list of slice dicts (from build_time_slices)
    title          : figure title
    figname        : stem for savefig
    extra_info_fn  : optional callable to add extra text to the legend
                     e.g. lambda sl: f"  N_dip={len(sl['df_dip']):,}"
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    # Zero reference line
    ax.axhline(0.0, color="grey", lw=1.0, ls="--", zorder=0)
    # Neutral shaded band ±0.05 around zero
    ax.axhspan(-0.05, 0.05, color="grey", alpha=0.07, zorder=0)

    plotted = 0
    for sl in time_slices:
        idx = sl["idx"]
        result = results_by_idx.get(idx)
        if result is None:
            continue  # skipped slice or too few points

        theta = result["theta"] * 60.0  # deg → arcmin
        w = np.asarray(result["w"], dtype=float)
        w_err = np.asarray(result["w_err"], dtype=float)
        w_err = np.where(np.isfinite(w_err), w_err, 0.0)

        color = SLICE_COLOR.get(idx, "black")
        marker = SLICE_MARKER.get(idx, "o")

        label = sl["label"]
        if extra_info_fn is not None:
            label += extra_info_fn(sl)

        ax.errorbar(
            theta,
            w,
            yerr=w_err,
            fmt=f"{marker}-",
            color=color,
            ms=5,
            lw=1.5,
            capsize=3,
            capthick=1.2,
            elinewidth=0.9,
            label=label,
            zorder=3,
        )
        plotted += 1

    if plotted == 0:
        ax.text(
            0.5,
            0.5,
            "No active slices with enough data",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=11,
        )

    ax.set_xscale("log")
    ax.set_xlabel(r"$\theta$ (arcmin)", fontsize=11)
    ax.set_ylabel(r"$w(\theta)$", fontsize=11)
    ax.set_title(
        f"{title}\n{FIELD_NAME}  |  slice width = {SLICE_DAYS} days  |  {plotted} active slices",
        fontsize=10,
    )
    ax.legend(
        loc="upper right",
        fontsize=7.5,
        framealpha=0.92,
        edgecolor="grey",
        ncol=1 if plotted <= 10 else 2,
        title="Time slice",
        title_fontsize=8,
    )
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("plot_slices_summary() defined.")

In [ ]:
# ── (A) Auto-correlation: all alerts ──────────────────────────────────────────
plot_slices_summary(
    results_acf_all,
    time_slices,
    title=r"Auto-correlation $w(\theta)$ — all DIA alerts",
    figname=f"TS_{FIELD_NAME}_acf_all_alerts",
    extra_info_fn=lambda sl: f"  (N={len(sl['df_all']):,})",
)

In [ ]:
# ── (B) Auto-correlation: dipole alerts only ──────────────────────────────────
plot_slices_summary(
    results_acf_dip,
    time_slices,
    title=r"Auto-correlation $w(\theta)$ — dipole alerts only",
    figname=f"TS_{FIELD_NAME}_acf_dipoles",
    extra_info_fn=lambda sl: f"  (N_dip={len(sl['df_dip']):,})",
)

In [ ]:
# ── (C) Cross-correlation non-dipoles × dipoles ───────────────────────────────
plot_slices_summary(
    results_xcf,
    time_slices,
    title=r"Cross-correlation $w(\theta)$ — non-dipoles $\times$ dipoles",
    figname=f"TS_{FIELD_NAME}_xcf_nodip_dip",
    extra_info_fn=lambda sl: f"  (N_nd={len(sl['df_ndip']):,} × N_dip={len(sl['df_dip']):,})",
)

## 8. Time evolution of $w(\theta)$ at a fixed angular scale

To visualise the **temporal trend** more directly, we extract $w$ at a
representative angular scale $\theta^*$ (configurable) and plot it as a
function of the slice mid-point MJD.  This is a convenient scalar summary
of the clustering evolution.

If $w(\theta^*)$ decreases with time for dipoles, it suggests that the AP
pipeline improved (fewer clustered dipole artefacts).

In [ ]:
# ── Angular scale at which to track the evolution ────────────────────────────
THETA_STAR_ARCMIN = 1.0  # representative scale in arcmin — change as needed


def extract_w_at_scale(
    results_by_idx: dict,
    theta_star_arcmin: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract w and w_err at the angular bin closest to *theta_star_arcmin*
    for each active slice, together with the slice mid-point MJD.

    Returns
    -------
    mjd_mids : array of slice mid-point MJDs
    w_vals   : w at theta_star for each slice (NaN if skipped)
    w_errs   : corresponding uncertainty
    """
    mjd_mids, w_vals, w_errs = [], [], []
    for sl in time_slices:
        mjd_mid = 0.5 * (sl["mjd_start"] + sl["mjd_end"])
        result = results_by_idx.get(sl["idx"])
        if result is None:
            mjd_mids.append(mjd_mid)
            w_vals.append(np.nan)
            w_errs.append(np.nan)
            continue
        theta_arcmin = result["theta"] * 60.0
        k = int(np.argmin(np.abs(theta_arcmin - theta_star_arcmin)))
        mjd_mids.append(mjd_mid)
        w_vals.append(float(result["w"][k]))
        w_errs.append(float(result["w_err"][k]) if np.isfinite(result["w_err"][k]) else 0.0)
    return np.array(mjd_mids), np.array(w_vals), np.array(w_errs)


mjd_m, w_all_t, e_all_t = extract_w_at_scale(results_acf_all, THETA_STAR_ARCMIN)
mjd_m, w_dip_t, e_dip_t = extract_w_at_scale(results_acf_dip, THETA_STAR_ARCMIN)
mjd_m, w_xcf_t, e_xcf_t = extract_w_at_scale(results_xcf, THETA_STAR_ARCMIN)

# Calendar date labels for x-axis ticks
date_labels = [mjd_to_datestr(m) for m in mjd_m]

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0.0, color="grey", lw=0.9, ls="--", zorder=0)

ax.errorbar(
    mjd_m, w_all_t, yerr=e_all_t, fmt="o-", color="steelblue", ms=6, lw=1.4, capsize=3, label="ACF all alerts"
)
ax.errorbar(
    mjd_m,
    w_dip_t,
    yerr=e_dip_t,
    fmt="s--",
    color="crimson",
    ms=6,
    lw=1.4,
    capsize=3,
    label="ACF dipoles only",
)
ax.errorbar(
    mjd_m,
    w_xcf_t,
    yerr=e_xcf_t,
    fmt="^:",
    color="seagreen",
    ms=6,
    lw=1.4,
    capsize=3,
    label=r"XCF non-dip $\times$ dip",
)

ax.set_xticks(mjd_m)
ax.set_xticklabels(date_labels, rotation=40, ha="right", fontsize=7)
ax.set_xlabel("Slice mid-point (date)", fontsize=10)
ax.set_ylabel(rf"$w(\theta^* = {THETA_STAR_ARCMIN}\,\mathrm{{arcmin}})$", fontsize=10)
ax.set_title(
    rf"{FIELD_NAME} — temporal evolution of $w$ at $\theta^*={THETA_STAR_ARCMIN}$ arcmin  "
    rf"(slice={SLICE_DAYS} days)",
    fontsize=10,
)
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
savefig(f"TS_{FIELD_NAME}_w_vs_time_at_{THETA_STAR_ARCMIN:.1f}arcmin")
plt.show()

## 9. Save per-slice results to parquet

In [ ]:
def save_slice_results(
    results_by_idx: dict,
    time_slices: list,
    label: str,  # e.g. 'acf_all', 'acf_dip', 'xcf'
) -> None:
    """Concatenate all slice results into one parquet with slice metadata columns."""
    rows = []
    for sl in time_slices:
        result = results_by_idx.get(sl["idx"])
        if result is None:
            continue
        n = len(result["theta"])
        rows.append(
            pd.DataFrame(
                {
                    "slice_idx": sl["idx"],
                    "mjd_start": sl["mjd_start"],
                    "mjd_end": sl["mjd_end"],
                    "date_start": sl["date_start"],
                    "date_end": sl["date_end"],
                    "theta_deg": result["theta"],
                    "theta_arcmin": result["theta"] * 60.0,
                    "w": result["w"],
                    "w_err": result["w_err"],
                    "field": FIELD_NAME,
                    "label": label,
                }
            )
        )
    if not rows:
        print(f"  [{label}] nothing to save.")
        return
    df_out = pd.concat(rows, ignore_index=True)
    path = os.path.join(DIR_DATA, f"wcf_{label}_{FIELD_NAME}_slices{SLICE_DAYS}d.parquet")
    df_out.to_parquet(path, index=False)
    print(f"  [{label}] saved {len(df_out):,} rows → {path}")


print("Saving results …")
save_slice_results(results_acf_all, time_slices, "acf_all")
save_slice_results(results_acf_dip, time_slices, "acf_dip")
save_slice_results(results_xcf, time_slices, "xcf")
print("Done.")

## 10. Notes and caveats

- **Empty slices** (fewer than `N_MIN_POINTS` alerts) are silently skipped.
  This naturally handles periods when the field is not accessible astronomically
  (e.g. COSMOS is below the horizon at night from Cerro Pachón in Northern autumn).
- **Slice width** (`SLICE_DAYS`) is a trade-off:
  - Too narrow → very few points per slice → noisy $w(\theta)$.
  - Too wide → temporal variation is washed out.
  - 30–60 days is a reasonable starting point for Rubin DDFs.
- **Catalogue size caveat**: the parquet from `01c` may be limited to `N_MAX=5000`
  alerts per DDF due to the API cap.  For COSMOS (high density), slices near the
  cap may not be representative.  Re-run `01c` with a larger `N_MAX` or the
  time-sliced API strategy if needed.
- The random catalogue is re-generated for every slice with the **same seed**,
  ensuring that differences between slices reflect only the data, not random noise
  in the reference catalogue.  Increase `N_RANDOM` for smoother $w(\theta)$.
